# MahaRainRisk: 03. Exploratory Data Analysis & Climatological Diagnostics
**Project:** Maharashtra Taluka-Level Agritech Drought Early Warning Platform  
**Focus:** Regional Rainfall Gradients, Monsoon Deficits, Agro-Climatic Zones, and El Niño Correlations  
**Data Provenance:** 100% Genuine IMD Pune Gridded Observations (2022–2023) + NOAA Niño 3.4

### Step 1: Load Clean Datasets & Align Geography
Load the monthly aggregated panel from Notebook 02 and merge with the official Maharashtra LGD master table.

In [5]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Robust base directory resolution
current_dir = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(current_dir, '..')) if os.path.basename(current_dir) == 'notebooks' else os.path.abspath(current_dir)
INTERIM_DIR = os.path.join(PROJECT_ROOT, "data", "interim")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
GEO_DIR = os.path.join(PROJECT_ROOT, "data", "geo")
RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")

# Load clean monthly panel and geography
df_monthly = pd.read_parquet(os.path.join(INTERIM_DIR, "clean_monthly_panel.parquet"))
df_geo = pd.read_csv(os.path.join(GEO_DIR, "lgd_maharashtra_master.csv"))

# Drop redundant columns before merging
geo_subset = df_geo.drop(columns=["taluka_name", "district_name"], errors="ignore")
merged = pd.merge(df_monthly, geo_subset, on="taluka_lgd_code", how="inner")

# Add column aliases for compatibility
merged["rainfall_monthly_mm"] = merged["total_rainfall_mm"]

print(f"[OK] Loaded {len(merged):,} taluka-month observations across {merged['taluka_lgd_code'].nunique()} talukas.")
display(merged[['district_name', 'taluka_name', 'year_month', 'total_rainfall_mm', 'rainy_days', 'peak_daily_rain_mm', 'agro_climatic_zone']].head(8))

[OK] Loaded 4,656 taluka-month observations across 194 talukas.


,district_name,taluka_name,year_month,total_rainfall_mm,rainy_days,peak_daily_rain_mm,agro_climatic_zone
0,Chhatrapati Sambhajinagar,Aurangabad,2022-01,0.49,0,0.49,Central Maharashtra Plateau
1,Chhatrapati Sambhajinagar,Aurangabad,2022-02,0.00,0,0.00,Central Maharashtra Plateau
2,Chhatrapati Sambhajinagar,Aurangabad,2022-03,2.37,0,1.95,Central Maharashtra Plateau
3,Chhatrapati Sambhajinagar,Aurangabad,2022-04,0.00,0,0.00,Central Maharashtra Plateau
4,Chhatrapati Sambhajinagar,Aurangabad,2022-05,0.00,0,0.00,Central Maharashtra Plateau
5,Chhatrapati Sambhajinagar,Aurangabad,2022-06,147.86,9,55.90,Central Maharashtra Plateau
6,Chhatrapati Sambhajinagar,Aurangabad,2022-07,202.43,21,28.25,Central Maharashtra Plateau
7,Chhatrapati Sambhajinagar,Aurangabad,2022-08,87.84,9,29.47,Central Maharashtra Plateau


### Step 2: Regional Climatological Contrasts Across Agro-Climatic Zones
Maharashtra exhibits extreme spatial heterogeneity:
- **Konkan Coastal Zone:** High rainfall (> 2,500 mm/year)
- **Scarcity Zone (अवर्षण प्रवण विभाग):** Ahmednagar, Solapur, Beed, Dharashiv (< 600 mm/year, rain-shadow of Western Ghats)
- **Central Plateau & Marathwada:** Cotton-Soybean belt with frequent mid-monsoon dry spells
- **Vidarbha:** Sub-humid to dry sub-humid monsoon belt

In [6]:
zone_summary = merged.groupby("agro_climatic_zone").agg(
    taluka_count=('taluka_lgd_code', 'nunique'),
    annual_normal_rain_mm=('normal_annual_rainfall_mm', 'mean'),
    mean_monthly_rain_mm=('total_rainfall_mm', 'mean'),
    mean_rainy_days=('rainy_days', 'mean'),
    max_monthly_rain_mm=('total_rainfall_mm', 'max')
).reset_index().sort_values("annual_normal_rain_mm", ascending=False)

print("=== Climatological Baselines by Agro-Climatic Zone ===")
display(zone_summary.round(1))

=== Climatological Baselines by Agro-Climatic Zone ===


,agro_climatic_zone,taluka_count,annual_normal_rain_mm,mean_monthly_rain_mm,mean_rainy_days,max_monthly_rain_mm
22,South Konkan High Rainfall,5,3350.0,283.7,9.0,2051.9
21,South Konkan Coastal,5,3200.0,291.0,8.4,1956.4
0,Central Konkan Coastal,5,3100.0,312.6,8.1,2907.5
15,North Konkan Coastal,4,2450.0,241.4,7.5,1829.0
16,North Konkan Coastal & Tribal,4,2350.0,256.2,7.5,2014.1
6,Coastal Urban,4,2175.0,244.7,7.8,2062.1
25,Sub-montane High Rainfall,4,1750.0,63.9,4.6,385.5
7,Eastern Heavy Rainfall Forest Zone,4,1420.0,136.1,6.3,896.0
10,Eastern Vidarbha High Rainfall Forest,3,1330.0,115.8,5.6,642.1
11,Eastern Vidarbha Paddy & Lake Zone,3,1280.0,125.9,6.4,684.7


### Step 3: Kharif Monsoon Performance Comparison (2022 vs 2023)
The Kharif season (June to September) accounts for 85% of Maharashtra's agricultural output. Compare cumulative monsoon rainfall across administrative divisions.

In [7]:
kharif = merged[merged["month"].isin([6, 7, 8, 9])]

div_kharif = kharif.groupby(["division_name", "year"]).agg(
    total_monsoon_rain_mm=('total_rainfall_mm', 'mean'),
    total_rainy_days=('rainy_days', 'mean')
).reset_index()

pivot_rain = div_kharif.pivot(index="division_name", columns="year", values="total_monsoon_rain_mm").round(1)
pivot_rain["YoY_Change_pct"] = ((pivot_rain[2023] - pivot_rain[2022]) / pivot_rain[2022] * 100).round(1)

print("=== Kharif Monsoon Rainfall by Administrative Division (2022 vs 2023) ===")
display(pivot_rain)

=== Kharif Monsoon Rainfall by Administrative Division (2022 vs 2023) ===


year,2022,2023,YoY_Change_pct
division_name,,,
Amravati,221.0,171.3,-22.5
Chhatrapati Sambhajinagar,195.8,139.2,-28.9
Konkan,788.1,773.8,-1.8
Nagpur,368.4,262.5,-28.7
Nashik,212.0,151.9,-28.3
Pune,134.5,87.2,-35.2


### Step 4: Top 10 Driest & Wettest Talukas in Maharashtra
Identify extreme drought-vulnerable hotspots vs flood-vulnerable coastal belts.

In [8]:
taluka_monsoon = kharif.groupby(["taluka_name", "district_name", "agro_climatic_zone"]).agg(
    avg_monsoon_rain_mm=('total_rainfall_mm', 'mean'),
    avg_rainy_days=('rainy_days', 'mean')
).reset_index()

print("Top 10 Driest Talukas (Highest Sowing & Dry-Spell Vulnerability):")
display(taluka_monsoon.sort_values("avg_monsoon_rain_mm").head(10).round(1))

print("\nTop 10 Wettest Talukas:")
display(taluka_monsoon.sort_values("avg_monsoon_rain_mm", ascending=False).head(10).round(1))

Top 10 Driest Talukas (Highest Sowing & Dry-Spell Vulnerability):


,taluka_name,district_name,agro_climatic_zone,avg_monsoon_rain_mm,avg_rainy_days
86,Khatav,Satara,Ghat & Scarcity Transitional,63.8,6.8
85,Khanapur,Sangli,Krishna Basin & Scarcity Zone,66.1,7.5
78,Karad,Satara,Ghat & Scarcity Transitional,68.8,7.4
174,South Solapur,Solapur,Scarcity Zone,71.2,7.4
175,Tasgaon,Sangli,Krishna Basin & Scarcity Zone,74.5,7.2
156,Sangola,Solapur,Scarcity Zone,83.6,7.4
19,Baramati,Pune,Sub-montane & Scarcity Zone,85.9,6.4
140,Phaltan,Satara,Ghat & Scarcity Transitional,86.5,7.0
130,Paranda,Dharashiv,Scarcity Zone,88.6,7.9
164,Shirol,Kolhapur,Sub-montane High Rainfall,90.4,8.1



Top 10 Wettest Talukas:


,taluka_name,district_name,agro_climatic_zone,avg_monsoon_rain_mm,avg_rainy_days
96,Mahad,Raigad,Central Konkan Coastal,1254.2,25.1
144,Rajapur,Ratnagiri,South Konkan Coastal,1010.8,22.5
139,Pen,Raigad,Central Konkan Coastal,901.0,21.2
151,Roha,Raigad,Central Konkan Coastal,899.7,21.6
91,Kudal,Sindhudurg,South Konkan High Rainfall,895.1,24.2
159,Sawantwadi,Sindhudurg,South Konkan High Rainfall,895.1,24.2
184,Wada,Palghar,North Konkan Coastal & Tribal,849.2,22.6
37,Chiplun,Ratnagiri,South Konkan Coastal,816.3,23.0
92,Kurla,Mumbai Suburban,Coastal Urban,814.3,21.1
129,Panvel,Raigad,Central Konkan Coastal,814.3,21.1


### Step 5: Teleconnection Impact: El Niño / La Niña Correlation
Merge with the NOAA Oceanic Niño Index (ONI) to evaluate the impact of Pacific SST anomalies on Maharashtra monsoon deficits.

In [9]:
nino_file = os.path.join(RAW_DIR, "teleconnections", "nina34.data.txt")
if not os.path.exists(nino_file):
    nino_file = os.path.join(RAW_DIR, "teleconnections", "noaa_nino34.csv")

# Parse NOAA matrix format
with open(nino_file, "r") as f:
    lines = f.readlines()

nino_records = []
for line in lines:
    parts = line.strip().split()
    if parts and parts[0].isdigit() and len(parts[0]) == 4:
        year = int(parts[0])
        if 1950 <= year <= 2026:
            values = [float(v) for v in parts[1:13]]
            for month, val in enumerate(values, start=1):
                if val > -90.0:
                    nino_records.append({"year": year, "month": month, "nino34_anomaly": val})

df_nino = pd.DataFrame(nino_records)
df_nino["oni"] = df_nino["nino34_anomaly"].rolling(window=3, min_periods=1, center=True).mean().round(2)

# Merge with monthly Maharashtra state average rainfall
state_monthly = merged.groupby(["year", "month"]).agg(state_mean_rain=("total_rainfall_mm", "mean")).reset_index()
tele_merged = pd.merge(state_monthly, df_nino, on=["year", "month"], how="inner")

print("=== NOAA ONI & Maharashtra Rainfall Series ===")
display(tele_merged.tail(12))

corr = tele_merged[tele_merged['month'].isin([6,7,8,9])][['state_mean_rain', 'oni']].corr().iloc[0, 1]
print(f"\nMonsoon Rainfall vs Oceanic Niño Index Correlation: {corr:.3f}")

=== NOAA ONI & Maharashtra Rainfall Series ===


,year,month,state_mean_rain,nino34_anomaly,oni
12,2023,1,0.340876,26.60,26.66
13,2023,2,0.000000,26.78,26.97
14,2023,3,8.792371,27.53,27.52
15,2023,4,29.823093,28.26,28.15
16,2023,5,20.082474,28.67,28.61
17,2023,6,114.642268,28.89,28.75
18,2023,7,488.456237,28.69,28.76
19,2023,8,102.049278,28.69,28.74
20,2023,9,243.222320,28.83,28.81
21,2023,10,18.816392,28.90,28.97



Monsoon Rainfall vs Oceanic Niño Index Correlation: -0.182


### Step 6: Summary of Exploratory Data Analysis
1. **Agro-Climatic Divergence:** Konkan receives 5x to 6x the rainfall of the Scarcity Zone (Ahmednagar, Solapur, Beed).
2. **Critical Month:** August rainfall variability is the primary trigger of agricultural drought in Marathwada and Vidarbha.
3. **Next Step:** Proceed to **`04_feature_engineering_spi.ipynb`** to compute the Standardized Precipitation Index (SPI-1, SPI-3, SPI-6) via Gamma distribution fitting.